# 05 · Deep Learning: Bottleneck Neural Network (Autoencoder-Inspired)

We build an **undercomplete network** — a neural network with a narrow
*bottleneck* hidden layer — that maps $(\\log_{10} x,\\, \\log_{10} Q^2)$
to $F_2^p$.

The bottleneck forces the model to compress the input information into a
low-dimensional latent representation before reconstructing the output.
Visualising this 2-D latent space shows how the network organises the
kinematic information internally.

This is **entirely new code** — architecture, training loop, and
visualisation are written from scratch for this project.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 120

from data_loader import load_lepton_dis, add_log_features, split_data, get_Xy, low_x_grid
from visualization import plot_coverage, plot_F2_vs_x, comparison_figure

DATA_DIR = "/Users/dikgarg/Desktop/Research/Neutrinos/postdoc/2025/ML/Data/Exp_data/LeptonDIS"

df_raw = load_lepton_dis(DATA_DIR)
df     = add_log_features(df_raw)
df_train, df_test = split_data(df, test_size=0.2, seed=42)

X_train, y_train = get_Xy(df_train)
X_test,  y_test  = get_Xy(df_test)

print(f"Training points: {len(X_train)}  |  Test points: {len(X_test)}")


In [ ]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

tf.random.set_seed(42)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype("float32")
X_test_s  = scaler.transform(X_test).astype("float32")
y_train_f = y_train.astype("float32")
y_test_f  = y_test.astype("float32")


## Network Architecture

In [ ]:
def build_bottleneck_net(input_dim=2, latent_dim=2,
                         encoder_units=(64, 32),
                         decoder_units=(32, 64)):
    """
    Input -> [encoder layers] -> Bottleneck (latent_dim) -> [decoder layers] -> Output

    The bottleneck layer is the compressed latent representation.
    """
    inp = tf.keras.Input(shape=(input_dim,), name="input")

    # Encoder path
    x = inp
    for i, u in enumerate(encoder_units):
        x = tf.keras.layers.Dense(u, activation="tanh",
                                  name=f"encoder_{i}")(x)

    latent = tf.keras.layers.Dense(latent_dim, activation="linear",
                                   name="bottleneck")(x)

    # Decoder path
    x = latent
    for i, u in enumerate(decoder_units):
        x = tf.keras.layers.Dense(u, activation="tanh",
                                  name=f"decoder_{i}")(x)

    output = tf.keras.layers.Dense(1, activation="linear",
                                   name="output")(x)

    full_model   = tf.keras.Model(inp, output,  name="bottleneck_net")
    encoder_only = tf.keras.Model(inp, latent,  name="encoder")

    return full_model, encoder_only


model, encoder = build_bottleneck_net(input_dim=2, latent_dim=2)
model.summary()


## Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-3),
    loss="mse",
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=50, restore_best_weights=True
)
lr_sched = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=20, min_lr=1e-5, verbose=0
)

history = model.fit(
    X_train_s, y_train_f,
    validation_data=(X_test_s, y_test_f),
    epochs=800,
    batch_size=64,
    callbacks=[early_stop, lr_sched],
    verbose=0,
)

print(f"Stopped at epoch {len(history.history['loss'])}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history.history["loss"],     label="Train loss")
ax.plot(history.history["val_loss"], label="Val loss", ls="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
ax.set_title("Training curve"); ax.legend(); ax.set_yscale("log")
ax.grid(True, ls="--", alpha=0.3)
plt.tight_layout()
plt.show()


## Test Set Performance

In [ ]:
y_pred_ae = model.predict(X_test_s, verbose=0).ravel()
print(f"Test MSE : {mean_squared_error(y_test, y_pred_ae):.5f}")
print(f"Test MAE : {mean_absolute_error(y_test, y_pred_ae):.5f}")
print(f"Test R2  : {r2_score(y_test, y_pred_ae):.4f}")


## Bottleneck (Latent) Space

In [ ]:
Z_all = encoder.predict(scaler.transform(df[["log10_x", "log10_Q2"]].values),
                       verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc1 = axes[0].scatter(Z_all[:, 0], Z_all[:, 1],
                      c=df["F2"].values, cmap="plasma", s=10, alpha=0.7)
plt.colorbar(sc1, ax=axes[0], label=r"$F_2^p$")
axes[0].set_xlabel("Latent dim 1"); axes[0].set_ylabel("Latent dim 2")
axes[0].set_title("Bottleneck representation — coloured by F2")

sc2 = axes[1].scatter(Z_all[:, 0], Z_all[:, 1],
                      c=df["log10_x"].values, cmap="viridis", s=10, alpha=0.7)
plt.colorbar(sc2, ax=axes[1], label=r"$\log_{10}(x)$")
axes[1].set_xlabel("Latent dim 1"); axes[1].set_ylabel("Latent dim 2")
axes[1].set_title(r"Bottleneck representation — coloured by $\log_{10}(x)$")

plt.tight_layout()
plt.savefig("../results/figures/05_latent_space.png", dpi=150)
plt.show()


## Low-x Extrapolation

In [ ]:
Q2_plot = [1.0, 5.0, 15.0, 30.0]
grids   = low_x_grid(x_min=1e-6, x_max=0.8, n_points=400, Q2_values=Q2_plot)

pred_ae = {"label": "Bottleneck NN", "color": "#4daf4a", "x_arr": None, "Q2_preds": {}}
for Q2v, (x_arr, X_feat) in grids.items():
    X_feat_s = scaler.transform(X_feat).astype("float32")
    pred_ae["x_arr"]         = x_arr
    pred_ae["Q2_preds"][Q2v] = model.predict(X_feat_s, verbose=0).ravel()

fig, _ = comparison_figure(Q2_plot, df, [pred_ae])
fig.suptitle("Bottleneck NN — predictions and low-x extrapolation", y=1.01)
plt.savefig("../results/figures/05_ae_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
